# 🏛️ The Procurement Desk: Autonomous Neuro-Symbolic Negotiation Engine
### Anveshan DataSprint 2.0 — 100.0% Exact Match Architecture (1.00000 on Leaderboard)

---

## 📌 Executive Summary & Architecture

In mission-critical enterprise procurement, deploying non-deterministic Large Language Models (LLMs) directly to make financial and legal negotiation decisions introduces three fatal failure modes:
1. **Mathematical Inconsistency:** LLMs cannot reliably perform decimal arithmetic, compound discounting, and exact financial rounding (round-half-up) across thousands of multi-currency negotiations.
2. **Adversarial Exploitation:** Generative models are vulnerable to indirect prompt injection, cross-language jailbreaks (Devanagari Hindi), and subtle contract traps (automatic renewals, supplier exclusivity, excessive upfront payments).
3. **Execution Latency & Cost:** Calling cloud LLM endpoints over thousands of turns incurs massive compute latency and API costs.

### 🛡️ The Neuro-Symbolic Deterministic Solution
This notebook implements an **ultra-efficient, deterministic neuro-symbolic engine** with zero external dependencies (pure Python standard library). It processes **2,000 negotiation threads in under 0.4 seconds (< 0.2 ms per thread)** with mathematical certitude and 100% playbook compliance:

- **Layer 1: Security & Legal Filter:** Evaluates prompt injections (multi-vector, HTML comment directives, Hindi jailbreaks) and scans for banned clauses (auto-renewal, >30% advance, unlimited liability, exclusivity).
- **Layer 2: Lexical Price Normalizer:** Resolves written numbers, discounts off previous quotes, corrections, unit pack conversions (LED 4-pack, paper reams), and contract-specific currency exchange rates.
- **Layer 3: Dynamic Policy Resolver:** Dynamically inherits category baselines, dated policy amendments, and bilateral supplier contracts using memoized lookup caches.
- **Layer 4: Formal Concession Engine:** Executes polynomial concession schedules $L(r) = \text{target} + (\text{walkaway} - \text{target}) \times ((r-1)/(R-1))^\beta$ with probation supplier restrictions and approval limit gates.


In [ ]:
import os
import sys
import re
import csv
import time
import glob
from decimal import Decimal, ROUND_HALF_UP

# Automatically resolve paths whether running on Kaggle or locally
KAGGLE_INPUT = '/kaggle/input/the-procurement-desk'
LOCAL_DIR = '.'

if os.path.exists(KAGGLE_INPUT):
    DATA_DIR = KAGGLE_INPUT
    OUTPUT_DIR = '/kaggle/working'
else:
    DATA_DIR = LOCAL_DIR
    OUTPUT_DIR = os.path.join(LOCAL_DIR, 'outputs')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data directory:   {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Data directory:   /kaggle/input/the-procurement-desk
Output directory: /kaggle/working


## 1. Playbook Reference Data & Dynamic Resolvers

Acme Corp negotiates across 15 spend categories with distinct parameters:
- **$R$**: Total allowed negotiation rounds (3, 4, or 5).
- **$\beta$**: Concession schedule exponent (1: linear, 2: quadratic, 3: cubic).
- **GST Rate**: Goods & Services Tax (5% catering, 12% paper, 18% services & hardware).
- **Billing Period**: Subscriptions quote per month or per year.


In [ ]:
# 15 Baseline Spend Categories
CATEGORIES = {
    'catering':        {'target': 280,    'walkaway': 340,     'rounds': 3, 'beta': 1, 'approval': 600000,   'gst': 0.05, 'billing': None},
    'cleaning':        {'target': 85000,  'walkaway': 100000,  'rounds': 4, 'beta': 2, 'approval': 900000,   'gst': 0.18, 'billing': 'month'},
    'cloud-servers':   {'target': 96000,  'walkaway': 115000,  'rounds': 5, 'beta': 3, 'approval': 6000000,  'gst': 0.18, 'billing': 'year'},
    'courier':         {'target': 160,    'walkaway': 200,     'rounds': 4, 'beta': 2, 'approval': 500000,   'gst': 0.18, 'billing': None},
    'crm-licenses':    {'target': 18000,  'walkaway': 22000,   'rounds': 4, 'beta': 2, 'approval': 2500000,  'gst': 0.18, 'billing': 'year'},
    'design-software': {'target': 30000,  'walkaway': 36000,   'rounds': 4, 'beta': 2, 'approval': 1500000,  'gst': 0.18, 'billing': 'year'},
    'headsets':        {'target': 3500,   'walkaway': 4200,    'rounds': 3, 'beta': 1, 'approval': 400000,   'gst': 0.18, 'billing': None},
    'laptops':         {'target': 62000,  'walkaway': 70000,   'rounds': 4, 'beta': 2, 'approval': 4000000,  'gst': 0.18, 'billing': None},
    'led-lighting':    {'target': 1800,   'walkaway': 2200,    'rounds': 3, 'beta': 1, 'approval': 350000,   'gst': 0.18, 'billing': None},
    'mobile-phones':   {'target': 24000,  'walkaway': 28000,   'rounds': 4, 'beta': 2, 'approval': 2000000,  'gst': 0.18, 'billing': None},
    'monitors':        {'target': 14000,  'walkaway': 16500,   'rounds': 3, 'beta': 1, 'approval': 1200000,  'gst': 0.18, 'billing': None},
    'office-chairs':   {'target': 9000,   'walkaway': 11000,   'rounds': 3, 'beta': 1, 'approval': 700000,   'gst': 0.18, 'billing': None},
    'printer-paper':   {'target': 2200,   'walkaway': 2600,    'rounds': 3, 'beta': 1, 'approval': 250000,   'gst': 0.12, 'billing': None},
    'security-guards': {'target': 21000,  'walkaway': 25000,   'rounds': 4, 'beta': 2, 'approval': 1000000,  'gst': 0.18, 'billing': 'month'},
    'training':        {'target': 6000,   'walkaway': 7500,    'rounds': 3, 'beta': 1, 'approval': 900000,   'gst': 0.18, 'billing': None},
}

CURRENCY_RATES = {'INR': 1.0, 'USD': 84.5, 'EUR': 91.0, 'GBP': 106.0}

CATEGORY_NAME_MAP = {
    'catering': 'catering', 'cleaning services': 'cleaning', 'cloud servers': 'cloud-servers',
    'courier services': 'courier', 'crm licenses': 'crm-licenses', 'design software': 'design-software',
    'headsets': 'headsets', 'laptops': 'laptops', 'led lighting': 'led-lighting',
    'mobile phones': 'mobile-phones', 'monitors': 'monitors', 'office chairs': 'office-chairs',
    'printer paper': 'printer-paper', 'security guards': 'security-guards', 'training workshops': 'training'
}

def normalize_category_name(raw_name):
    clean = raw_name.strip().lower()
    return CATEGORY_NAME_MAP.get(clean, clean.replace(' ', '-'))

def load_playbook_amendments_and_suppliers(base_dir):
    amendments = []
    supplier_status = {}
    supplier_price_caps = {}
    supplier_fx = {}

    candidates = [
        os.path.join(base_dir, 'playbook', 'playbook'),
        os.path.join(base_dir, 'playbook'),
        base_dir
    ]
    pb_dir = next((c for c in candidates if os.path.exists(os.path.join(c, 'amendments'))), None)

    if pb_dir:
        for fpath in glob.glob(os.path.join(pb_dir, 'amendments', '*.md')):
            aid = os.path.splitext(os.path.basename(fpath))[0]
            with open(fpath, encoding='utf-8') as f:
                content = f.read()
            m_date = re.search(r'\*\*Effective from:\*\*\s*(\d{4}-\d{2}-\d{2})', content)
            effective_date = m_date.group(1) if m_date else ''
            m_change = re.search(
                r'the\s+(target price|walk-away price|approval limit)\s+for\s+(.*?)\s+is\s+now\s+[₹Rs\.]*\s*([\d,]+)',
                content, re.I
            )
            if m_change:
                field_raw, cat_raw, val_raw = m_change.group(1).lower(), m_change.group(2).strip(), m_change.group(3).replace(',', '')
                field = 'target' if 'target' in field_raw else ('walkaway' if 'walk' in field_raw else 'approval')
                amendments.append({
                    'id': aid, 'category': normalize_category_name(cat_raw),
                    'date': effective_date, 'field': field, 'value': int(val_raw)
                })
        amendments.sort(key=lambda a: a['date'])

        for fpath in glob.glob(os.path.join(pb_dir, 'suppliers', '*.md')):
            with open(fpath, encoding='utf-8') as f:
                content = f.read()
            m_name = re.search(r'#\s*Supplier:\s*(.*)', content)
            if not m_name:
                continue
            sname = m_name.group(1).strip()
            m_status = re.search(r'\*\*Status:\*\*\s*(Approved|BLACKLISTED|PROBATION)', content, re.I)
            if m_status:
                status = m_status.group(1).upper()
                if status in ('BLACKLISTED', 'PROBATION'):
                    supplier_status[sname] = status
            m_fx = re.search(r'invoices\s+in\s+([A-Z]{3})\s+are\s+converted\s+at\s+[₹Rs\.]*\s*([\d.]+)\s+per\s+[A-Z]{3}', content, re.I)
            if m_fx:
                cur, rate = m_fx.group(1).upper(), float(m_fx.group(2))
                supplier_fx.setdefault(sname, {})[cur] = rate
            m_cap = re.search(r'price\s+cap\s+for\s+(.*?)\s+of\s+[₹Rs\.]*\s*([\d,]+)\s+per', content, re.I)
            if m_cap:
                cat, cap_val = normalize_category_name(m_cap.group(1).strip()), int(m_cap.group(2).replace(',', ''))
                supplier_price_caps.setdefault(sname, {})[cat] = cap_val

    return amendments, supplier_status, supplier_price_caps, supplier_fx

AMENDMENTS, SUPPLIER_STATUS, SUPPLIER_PRICE_CAPS, SUPPLIER_FX = load_playbook_amendments_and_suppliers(DATA_DIR)
print(f"Loaded {len(AMENDMENTS)} amendments, {len(SUPPLIER_STATUS)} special suppliers, {len(SUPPLIER_PRICE_CAPS)} price caps, {len(SUPPLIER_FX)} FX overrides.")


Loaded 5 amendments, 14 special suppliers, 4 price caps, 4 FX overrides.


## 2. Ultra-Fast Security & Legal Screening
- **Multi-Vector Prompt Injection Defense:** Pre-filters scan for keywords before executing the compiled regex engine. Catches AI directives, non-English prompts (Devanagari Hindi), HTML comment injections, and spoofed approvals. Legitimate business mentions (*"assistant manager"*, *"AI route planning"*, *"ERP system"*) remain safely permitted per Acme's security protocol.
- **Banned Contract Terms:** Scans the supplier's latest message for auto-renewal clauses, advance payments exceeding 30% (including phrased numbers like *"half upfront"*), unlimited liability, and supplier exclusivity (*"exclusive supplier"*, *"buy only from us"*).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SECURITY: PROMPT INJECTION DETECTION (OPTIMIZED)
# ═══════════════════════════════════════════════════════════════════════════════

_INJECTION_PATTERNS = [
    r'<!--.*?(?:assistant|approve|accept|reject|escalate|mark|grant|override).*?-->',
    r'SYSTEM\s+NOTE\s+TO\s+AI',
    r'note\s+for\s+the\s+language\s+model',
    r'note\s+to\s+(?:the\s+)?(?:AI|bot|model|automated)',
    r'(?:if\s+)?(?:an?\s+)?automated\s+system\s+is\s+reading',
    r'(?:if\s+)?(?:an?\s+)?(?:AI|bot|model)\s+(?:is\s+)?(?:reading|processing|reviewing|screens)',
    r'whichever\s+bot',
    r'ignore\s+(?:all\s+)?(?:your\s+)?previous\s+instructions',
    r'\[admin\s+override\]|admin\s+override',
    r'skip\s+(?:the\s+)?playbook\s+checks',
    r'AI\s*सहायक|सहायक\s+के\s+लिए\s+निर्देश|निर्देश.*?ACCEPT',
    r'(?:pre-?approved|already\s+approved|approval\s+(?:already|has\s+been)\s+granted)',
    r'reviewing\s+assistant\s+should\s+accept',
    r'our\s+CFO\s+already\s+signed\s+off',
    r'(?:output|return|set|mark|record)\s+(?:this\s+)?(?:thread\s+)?(?:as\s+)?(?:ACCEPT|REJECT|COUNTER|ESCALATE)',
    r'(?:record|set|mark)\s+the\s+decision\s+as',
    r'just\s+mark\s+it\s+accepted',
    r'(?:dear|to|for)\s+(?:the\s+)?(?:AI|bot|automated)\s+(?:agent|reviewer|system|reader)',
    r'please\s+(?:approve|accept|grant|override)',
]
_COMBINED_INJECTION_RE = re.compile('|'.join(f'(?:{p})' for p in _INJECTION_PATTERNS), re.I | re.DOTALL)

def detect_injection(full_msg):
    m_lower = full_msg.lower()
    if not (
        '<!--' in full_msg or
        'system note' in m_lower or
        'language model' in m_lower or
        'note to' in m_lower or
        'automated system' in m_lower or
        'whichever bot' in m_lower or
        'previous instructions' in m_lower or
        'ignore' in m_lower or
        'admin override' in m_lower or
        'playbook' in m_lower or
        'सहायक' in full_msg or 'निर्देश' in full_msg or
        'pre-approved' in m_lower or 'preapproved' in m_lower or
        'already approved' in m_lower or 'signed off' in m_lower or
        'has been granted' in m_lower or 'approval already' in m_lower or
        'reviewing assistant' in m_lower or
        'decision as' in m_lower or 'mark it accepted' in m_lower or
        (('reading' in m_lower or 'processing' in m_lower or 'screens' in m_lower or 'reviewing' in m_lower) and ('ai' in m_lower or 'bot' in m_lower or 'model' in m_lower)) or
        (('dear' in m_lower or 'to' in m_lower or 'for' in m_lower) and ('ai' in m_lower or 'bot' in m_lower or 'automated' in m_lower)) or
        'please approve' in m_lower or 'please accept' in m_lower or 'please grant' in m_lower or 'please override' in m_lower or
        ('accept' in full_msg and ('output' in m_lower or 'return' in m_lower or 'set' in m_lower or 'mark' in m_lower or 'record' in m_lower))
    ):
        return False
    return bool(_COMBINED_INJECTION_RE.search(full_msg))

# ═══════════════════════════════════════════════════════════════════════════════
#  LEGAL: BANNED CONTRACT TERMS (OPTIMIZED)
# ═══════════════════════════════════════════════════════════════════════════════

RE_LATEST_MARKER = re.compile(r'\[Latest message\s*[—–-]')

def isolate_latest_message(full_msg):
    marker = RE_LATEST_MARKER.search(full_msg)
    if marker:
        rest = full_msg[marker.start():]
        newline = rest.find('\n')
        return rest[newline + 1:] if newline != -1 else rest
    return full_msg

RE_DENIAL = re.compile(
    r'(?:no|not|without)\s+(?:an?\s+)?auto(?:matic(?:ally)?)?[\s-]*renew|'
    r'renewal\s+only\s+(?:by\s+mutual|if\s+both)',
    re.I
)
RE_AUTO_RENEW = re.compile(r'auto(?:matic(?:ally)?)?[\s-]*renew', re.I)
RE_YEAR_TO_YEAR = re.compile(r'continues?\s+(?:from\s+)?year\s+to\s+year\s+unless', re.I)
RE_RENEWS_OWN = re.compile(r'renews?\s+on\s+its\s+own', re.I)
RE_HALF_ADVANCE = re.compile(r'\bhalf\b.*?(?:upfront|advance)|(?:upfront|advance).*?\bhalf\b', re.I)
RE_PCT_ADVANCE = re.compile(r'(\d+)\s*%.*?(?:upfront|advance)|(?:upfront|advance).*?(\d+)\s*%', re.I)
RE_LIABILITY = re.compile(r'liability.*?unlimited|unlimited.*?liability', re.I)
RE_NO_EXCL = re.compile(r'(?:no|not|without)\s+(?:asking\s+for\s+)?exclusiv', re.I)
RE_EXCL = re.compile(r'exclusive\s+supplier|exclusivity|only\s+from\s+us|must\s+buy\s+only\s+from|buy\s+these\s+items\s+only\s+from', re.I)

def detect_banned_terms(latest_msg):
    text = latest_msg
    t_lower = text.lower()

    if 'renew' in t_lower or 'year to year' in t_lower:
        if not RE_DENIAL.search(text):
            if RE_AUTO_RENEW.search(text) or RE_YEAR_TO_YEAR.search(text) or RE_RENEWS_OWN.search(text):
                return True

    if 'upfront' in t_lower or 'advance' in t_lower or 'half' in t_lower:
        if RE_HALF_ADVANCE.search(text):
            return True
        for m in RE_PCT_ADVANCE.finditer(text):
            pct = int(m.group(1) or m.group(2))
            if pct > 30:
                return True

    if 'liability' in t_lower and 'unlimited' in t_lower:
        if RE_LIABILITY.search(text):
            return True

    if 'exclusiv' in t_lower or 'only from' in t_lower:
        if not RE_NO_EXCL.search(text):
            if RE_EXCL.search(text):
                return True

    return False


## 3. High-Speed Lexical Price Extraction & Normalization
Supplier offers are parsed and normalized strictly to **INR, excluding GST, per category base unit**:
- **Worded Numbers:** Converts written English numbers (*"twenty-six thousand four hundred"*) to integers.
- **Discounts & Corrections:** Handles explicit corrections (*"CORRECTION — ... should read ₹X"*), list price discounts (*"15% off list price"*), and previous quote discounts (inheriting earlier subscription billing periods).
- **Sub-units & Packs:** Converts 1 pack = 4 fixtures (LED lighting), 1 box = 10 reams (Printer paper), and per dozen (÷ 12).
- **Modifiers:** Free items (*"1 free for every 10 you pay"*) and volume discounts above quantity thresholds.


In [ ]:
_CUR = r'(?:₹|\bRs\.?\s*|\bINR\s+|\bUSD\s*|\$|\bEUR\s*|€|\bGBP\s*|£|\bSGD\s*|\bAED\s*|\bJPY\s*)'
_AMT = r'(\d[\d,]*(?:\.\d+)?)'

def parse_number(s):
    return float(s.strip().replace(',', ''))

_UNITS = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
    'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14,
    'fifteen': 15, 'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19,
}
_TENS = {
    'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50,
    'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90,
}

def words_to_number(text):
    text = text.lower().replace('-', ' ').replace(' and ', ' ')
    tokens = text.split()
    current, result = 0, 0
    for tok in tokens:
        if tok in _UNITS:
            current += _UNITS[tok]
        elif tok in _TENS:
            current += _TENS[tok]
        elif tok == 'hundred':
            current = (current or 1) * 100
        elif tok == 'thousand':
            result += (current or 1) * 1000
            current = 0
        elif tok == 'lakh':
            result += (current or 1) * 100000
            current = 0
        elif tok == 'crore':
            result += (current or 1) * 10000000
            current = 0
        else:
            break
    result += current
    return result if result > 0 else None

def detect_currency(prefix_text):
    t = prefix_text.strip().lower()
    if '₹' in t or 'rs' in t or 'inr' in t: return 'INR'
    if '$' in t or 'usd' in t: return 'USD'
    if '€' in t or 'eur' in t: return 'EUR'
    if '£' in t or 'gbp' in t: return 'GBP'
    if 'sgd' in t: return 'SGD'
    if 'aed' in t: return 'AED'
    if 'jpy' in t: return 'JPY'
    return None

RE_GST = re.compile(r'inclu(?:sive|ding)\s+(?:of\s+)?GST|incl\.?\s*GST|GST\s+included', re.I)
def is_gst_inclusive(text):
    if 'gst' not in text.lower():
        return False
    return bool(RE_GST.search(text))

RE_PERIOD_MONTH = re.compile(r'per\s+(?:\w+\s+)?per\s+month\b|per\s+month\b|\bmonthly\b', re.I)
RE_PERIOD_YEAR = re.compile(r'per\s+(?:\w+\s+)?per\s+year\b|per\s+year\b|per\s+annum\b|\bannual\b|\byearly\b', re.I)

def detect_billing_period(text):
    t_lower = text.lower()
    if 'month' in t_lower:
        if RE_PERIOD_MONTH.search(text): return 'month'
    if 'year' in t_lower or 'annum' in t_lower or 'annual' in t_lower:
        if RE_PERIOD_YEAR.search(text): return 'year'
    return None

# Pre-compiled Price Extraction Regexes
RE_CORRECTION_1 = re.compile(r'CORRECTION.*?should\s+read\s+(' + _CUR + r')\s*' + _AMT, re.I | re.DOTALL)
RE_CORRECTION_2 = re.compile(r'CORRECTION.*?(' + _CUR + r')\s*' + _AMT, re.I | re.DOTALL)
RE_TOTAL = re.compile(r'(' + _CUR + r')\s*' + _AMT + r'\s+in\s+total', re.I)
RE_LIST_DISC = re.compile(r'(\d+)\s*%\s*(?:discount|off)\s+(?:on\s+)?(?:our\s+)?list\s+price\s+of\s+(' + _CUR + r')\s*' + _AMT, re.I)
RE_PREV_DISC = re.compile(r'(?:a\s+)?(?:further\s+)?(\d+)\s*%\s*off\s+(?:our\s+)?previous\s+(?:quote|price|offer)', re.I)
RE_RANGE = re.compile(r'between\s+(' + _CUR + r')\s*' + _AMT + r'\s+and\s+(?:' + _CUR + r')?\s*' + _AMT, re.I)
RE_TIER = re.compile(
    r'(' + _CUR + r')\s*' + _AMT + r'\s+per\s+\w+.*?for\s+orders?\s+of\s+up\s+to\s+(\d+).*?'
    r'(?:' + _CUR + r')?\s*' + _AMT + r'.*?(?:for\s+larger|per\s+\w+\s+for\s+larger)',
    re.I | re.DOTALL
)
RE_K_LAKH = re.compile(r'(' + _CUR + r')\s*(\d+(?:\.\d+)?)\s*(k|lakh)\b', re.I)
RE_DOZEN = re.compile(r'(' + _CUR + r')\s*' + _AMT + r'\s+per\s+dozen', re.I)
_NUM_WORD = r'(?:zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand|lakh|crore|and)'
RE_WORDS = re.compile(r'((?:' + _NUM_WORD + r'[\s-]*){2,})\s*rupees?\s+per\s+', re.I)
RE_DIRECT = re.compile(r'(' + _CUR + r')\s*' + _AMT + r'\s+per\s+', re.I)
RE_FREE = re.compile(r'plus\s+(\d+)\s+free\s+\w+\s+for\s+every\s+(\d+)\s+you\s+pay', re.I)
RE_VOL = re.compile(r'(?:a\s+)?(?:further\s+)?(\d+)\s*%\s*off\s+for\s+orders?\s+(?:of\s+)?(\d+)\s*(?:\w+\s+)?(?:or\s+more|and\s+above|\+)', re.I)
RE_PACK = re.compile(r'\bpack\b', re.I)
RE_REAM = re.compile(r'\breams?\b', re.I)
RE_PER_N = re.compile(r'per\s+(\d+)\s+(?:fixtures|shipments|meals|units|seats|phones|monitors|chairs|boxs|boxes|guards|participants|sites|laptops|headsets)', re.I)
RE_EARLIER_QUOTE = re.compile(r'(?:₹|Rs\.?\s*|INR\s+)' + _AMT, re.I)

_SUPPLIER_EARLIER_RE_CACHE = {}
def get_supplier_earlier_re(supplier_name):
    pat = _SUPPLIER_EARLIER_RE_CACHE.get(supplier_name)
    if pat is None:
        pat = re.compile(r'\[Earlier message\s*[—–-]\s*' + re.escape(supplier_name) + r'.*?\](.*?)(?=\[|$)', re.DOTALL)
        _SUPPLIER_EARLIER_RE_CACHE[supplier_name] = pat
    return pat

def extract_earlier_supplier_quote(full_msg, supplier_name):
    pattern = get_supplier_earlier_re(supplier_name)
    quotes = []
    for m in pattern.finditer(full_msg):
        pm = RE_EARLIER_QUOTE.search(m.group(1))
        if pm:
            quotes.append(parse_number(pm.group(1)))
    return quotes[-1] if quotes else None

def extract_and_normalize_price(latest_msg, full_msg, category_key, quantity, supplier_name,
                                 gst_rate, billing_category, supplier_fx, currency_rates):
    text = latest_msg
    t_lower = text.lower()
    gst_incl = is_gst_inclusive(text)
    bill_period = detect_billing_period(text)

    raw_price = None
    currency = None
    is_total = False
    is_per_dozen = False
    is_prev_quote = False

    # 1. Correction
    if 'correction' in t_lower or 'should read' in t_lower:
        m = RE_CORRECTION_1.search(text) or RE_CORRECTION_2.search(text)
        if m:
            currency = detect_currency(m.group(1))
            raw_price = parse_number(m.group(2))

    # 2. Total
    if raw_price is None and 'total' in t_lower:
        m = RE_TOTAL.search(text)
        if m:
            currency = detect_currency(m.group(1))
            raw_price = parse_number(m.group(2))
            is_total = True

    # 3. List Price Discount
    if raw_price is None and 'list price' in t_lower:
        m = RE_LIST_DISC.search(text)
        if m:
            discount_pct = int(m.group(1))
            currency = detect_currency(m.group(2))
            list_price = parse_number(m.group(3))
            raw_price = list_price * (1 - discount_pct / 100.0)

    # 4. Previous Quote Discount
    if raw_price is None and ('previous' in t_lower or 'prior' in t_lower or 'earlier' in t_lower):
        m = RE_PREV_DISC.search(text)
        if m:
            discount_pct = int(m.group(1))
            prev = extract_earlier_supplier_quote(full_msg, supplier_name)
            if prev is not None:
                raw_price = prev * (1 - discount_pct / 100.0)
                currency = 'INR'
                is_prev_quote = True

    # 5. Price Range
    if raw_price is None and 'between' in t_lower:
        m = RE_RANGE.search(text)
        if m:
            currency = detect_currency(m.group(1))
            raw_price = max(parse_number(m.group(2)), parse_number(m.group(3)))

    # 6. Tiered Pricing
    if raw_price is None and ('larger' in t_lower or 'up to' in t_lower):
        m = RE_TIER.search(text)
        if m:
            currency = detect_currency(m.group(1))
            price_low = parse_number(m.group(2))
            threshold = int(m.group(3))
            price_high_tier = parse_number(m.group(4))
            raw_price = price_low if quantity <= threshold else price_high_tier

    # 7. K and Lakh
    if raw_price is None and ('k' in t_lower or 'lakh' in t_lower):
        m = RE_K_LAKH.search(text)
        if m:
            currency = detect_currency(m.group(1))
            val, unit = float(m.group(2)), m.group(3).lower()
            raw_price = val * 1000.0 if unit == 'k' else val * 100000.0

    # 8. Per Dozen
    if raw_price is None and 'dozen' in t_lower:
        m = RE_DOZEN.search(text)
        if m:
            currency = detect_currency(m.group(1))
            raw_price = parse_number(m.group(2))
            is_per_dozen = True

    # 9. Words
    if raw_price is None and ('rupee' in t_lower or 'rupees' in t_lower):
        m = RE_WORDS.search(text)
        if m:
            num = words_to_number(m.group(1))
            if num:
                raw_price = float(num)
                currency = 'INR'

    # 10. Direct Price
    if raw_price is None:
        m = RE_DIRECT.search(text)
        if m:
            currency = detect_currency(m.group(1))
            raw_price = parse_number(m.group(2))

    if raw_price is None:
        return None, 'UNKNOWN'

    if currency is None:
        currency = 'INR'

    if billing_category is not None and not is_total and not is_prev_quote and bill_period is None:
        return None, 'UNKNOWN'

    price = raw_price

    if 'free' in t_lower:
        m_free = RE_FREE.search(text)
        if m_free:
            free_u, paid_u = int(m_free.group(1)), int(m_free.group(2))
            price = price * paid_u / (paid_u + free_u)

    if '%' in text and ('off' in t_lower or 'discount' in t_lower):
        m_vol = RE_VOL.search(text)
        if m_vol:
            if quantity >= int(m_vol.group(2)):
                price = price * (1 - int(m_vol.group(1)) / 100.0)

    if category_key == 'led-lighting' and RE_PACK.search(text):
        price = price / 4.0
    elif category_key == 'printer-paper' and RE_REAM.search(text):
        price = price * 10.0

    m_per_n = RE_PER_N.search(text)
    if m_per_n:
        price = price / float(m_per_n.group(1))

    if is_total:
        price = price / quantity
    if is_per_dozen:
        price = price / 12.0

    if currency != 'INR':
        fx = supplier_fx.get(supplier_name, {}).get(currency, currency_rates.get(currency))
        if fx is None:
            return None, currency
        price = price * fx

    if gst_incl:
        price = price / (1 + gst_rate)

    if billing_category is not None and bill_period is not None:
        if billing_category == 'year' and bill_period == 'month':
            price = price * 12
        elif billing_category == 'month' and bill_period == 'year':
            price = price / 12

    return price, currency


## 4. Playbook Decision Protocol

Rules are executed strictly in order (first rule to fire wins):
1. **Missing Information:** No usable number, unlisted currency (`SGD`, `AED`, `JPY`), or subscription missing billing period $\to$ `ESCALATE:MISSING_INFO:`.
2. **Blacklisted Supplier:** Status is `BLACKLISTED` $\to$ `REJECT:BLACKLISTED_SUPPLIER:`.
3. **Prompt Injection:** Untrusted message instructing system $\to$ `ESCALATE:PROMPT_INJECTION:`.
4. **Banned Contract Terms:** Unacceptable clauses in latest offer $\to$ `ESCALATE:BANNED_TERM:`.
5. **Price Decision:** Compare normalized offer $p$ with concession ceiling $L(r)$:
   $$L(r) = \text{target} + (\text{walkaway} - \text{target}) \times \left(\frac{r - 1}{R - 1}\right)^\beta$$
   - For probation vendors: $L(r) = \text{target}$ in all rounds.
   - If $p \le L(r) \to$ Step 6.
   - If $p > L(r)$ on final round ($r \ge R$) $\to$ `REJECT:ABOVE_WALKAWAY:`.
   - Otherwise $\to$ `COUNTER:COUNTER_OFFER:<rounded L(r)>` (round half-up).
6. **Approval Limit:** If acceptable offer total $p \times \text{quantity} > \text{approval limit} \to$ `ESCALATE:OVER_APPROVAL_LIMIT:`. Otherwise $\to$ `ACCEPT:ACCEPTABLE_PRICE:`.


In [ ]:
_POLICY_CACHE = {}

def get_effective_policy(category_key, thread_date, supplier_name):
    cache_key = (category_key, thread_date, supplier_name)
    if cache_key in _POLICY_CACHE:
        return _POLICY_CACHE[cache_key]

    base = CATEGORIES[category_key]
    policy = {
        'target': base['target'], 'walkaway': base['walkaway'], 'rounds': base['rounds'],
        'beta': base['beta'], 'approval': base['approval'], 'gst': base['gst'], 'billing': base['billing'],
    }
    for amd in AMENDMENTS:
        if amd['category'] == category_key and amd['date'] <= thread_date:
            policy[amd['field']] = amd['value']

    if supplier_name in SUPPLIER_PRICE_CAPS:
        caps = SUPPLIER_PRICE_CAPS[supplier_name]
        if category_key in caps:
            policy['walkaway'] = caps[category_key]

    _POLICY_CACHE[cache_key] = policy
    return policy

def compute_L(r, policy, is_probation):
    if is_probation:
        return float(policy['target'])
    R, beta = policy['rounds'], policy['beta']
    target, walkaway = policy['target'], policy['walkaway']
    if R == 1:
        return float(target)
    return target + (walkaway - target) * (((r - 1) / (R - 1)) ** beta)

def round_half_up(value):
    return int(Decimal(str(value)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def decide(thread):
    supplier, category, date = thread['supplier'], thread['category'], thread['date']
    quantity, round_num, message = int(thread['quantity']), int(thread['round']), thread['supplier_message']

    policy = get_effective_policy(category, date, supplier)
    is_probation = (SUPPLIER_STATUS.get(supplier) == 'PROBATION')
    latest = isolate_latest_message(message)

    # 1. Missing Info Check
    price, cur = extract_and_normalize_price(
        latest, message, category, quantity, supplier,
        policy['gst'], policy['billing'], SUPPLIER_FX, CURRENCY_RATES
    )
    if price is None:
        return 'ESCALATE:MISSING_INFO:'

    # 2. Blacklisted Supplier
    if SUPPLIER_STATUS.get(supplier) == 'BLACKLISTED':
        return 'REJECT:BLACKLISTED_SUPPLIER:'

    # 3. Prompt Injection
    if detect_injection(message):
        return 'ESCALATE:PROMPT_INJECTION:'

    # 4. Banned Contract Terms
    if detect_banned_terms(latest):
        return 'ESCALATE:BANNED_TERM:'

    # 5. Price Decision
    L_r = compute_L(round_num, policy, is_probation)
    R = policy['rounds']

    if price <= L_r:
        if price * quantity > policy['approval']:
            return 'ESCALATE:OVER_APPROVAL_LIMIT:'
        return 'ACCEPT:ACCEPTABLE_PRICE:'
    else:
        if round_num >= R:
            return 'REJECT:ABOVE_WALKAWAY:'
        return f'COUNTER:COUNTER_OFFER:{round_half_up(L_r)}'


## 5. Ground Truth Verification on Sample Dataset (40/40 Exact Match)

In [ ]:
sample_threads_path = os.path.join(DATA_DIR, 'sample_threads.csv')
sample_answers_path = os.path.join(DATA_DIR, 'sample_answers.csv')

if os.path.exists(sample_threads_path) and os.path.exists(sample_answers_path):
    with open(sample_threads_path, encoding='utf-8') as f:
        samples = list(csv.DictReader(f))
    with open(sample_answers_path, encoding='utf-8') as f:
        answers = {r['thread_id']: r['label'] for r in csv.DictReader(f)}

    correct = 0
    for s in samples:
        pred = decide(s)
        expected = answers.get(s['thread_id'])
        if pred == expected:
            correct += 1
        else:
            print(f"Mismatch {s['thread_id']}: Expected {expected}, Got {pred}")

    print(f"\n{'='*60}")
    print(f"  BENCHMARK EXACT MATCH: {correct}/{len(samples)} ({100*correct/len(samples):.1f}%)")
    print(f"{'='*60}\n")
    assert correct == len(samples), "Benchmark verification failed!"



  BENCHMARK EXACT MATCH: 40/40 (100.0%)



## 6. Execution on Full Test Set (2,000 Threads) & Runtime Profiling

In [ ]:
threads_path = os.path.join(DATA_DIR, 'threads.csv')
submission_path = os.path.join(OUTPUT_DIR, 'submission.csv')

with open(threads_path, encoding='utf-8') as f:
    threads = list(csv.DictReader(f))

print(f"Processing {len(threads)} test threads...")
t0 = time.perf_counter()

results = []
dist = {}

for t in threads:
    label = decide(t)
    results.append({'thread_id': t['thread_id'], 'label': label})
    action_reason = ':'.join(label.split(':')[:2])
    dist[action_reason] = dist.get(action_reason, 0) + 1

elapsed = time.perf_counter() - t0

# Write submission.csv
with open(submission_path, 'w', newline='', encoding='utf-8') as f:
    f.write('thread_id,label\n')
    for r in results:
        f.write(f"{r['thread_id']},{r['label']}\n")

print(f"\n⚡ Processed {len(results)} threads in {elapsed:.3f}s ({elapsed/len(threads)*1000:.2f} ms/thread)!")
print(f"✅ Successfully exported predictions to {submission_path}")
print("\n--- Final Decision Distribution (2,000 Rows) ---")
for k, v in sorted(dist.items()):
    print(f"  {k:<32}: {v:>4} ({v/len(threads)*100:.1f}%)")


Processing 2000 test threads...

⚡ Processed 2000 threads in 0.375s (0.19 ms/thread)!
✅ Successfully exported predictions to /kaggle/working/submission.csv

--- Final Decision Distribution (2,000 Rows) ---
  ACCEPT:ACCEPTABLE_PRICE          :  359 (18.0%)
  COUNTER:COUNTER_OFFER            :  627 (31.3%)
  ESCALATE:BANNED_TERM             :  144 (7.2%)
  ESCALATE:MISSING_INFO            :   92 (4.6%)
  ESCALATE:OVER_APPROVAL_LIMIT     :  286 (14.3%)
  ESCALATE:PROMPT_INJECTION        :  160 (8.0%)
  REJECT:ABOVE_WALKAWAY            :  227 (11.3%)
  REJECT:BLACKLISTED_SUPPLIER      :  105 (5.2%)


## 7. Submission Contract Validation

In [ ]:
pattern = r'^(ACCEPT|COUNTER|REJECT|ESCALATE):[A-Z0-9_]+:[0-9]*$'
with open(submission_path, encoding='utf-8') as f:
    sub_rows = list(csv.DictReader(f))

assert len(sub_rows) == 2000, f"Expected 2000 rows, got {len(sub_rows)}"

for r in sub_rows:
    lbl = r['label']
    assert re.match(pattern, lbl), f"Invalid label format: {lbl}"
    parts = lbl.split(':')
    if parts[0] == 'COUNTER':
        assert parts[2].isdigit(), f"Missing counter price: {lbl}"
    else:
        assert parts[2] == '', f"Unexpected counter price: {lbl}"

print("✅ All 2,000 rows strictly comply with competition submission specifications.")


✅ All 2,000 rows strictly comply with competition submission specifications.
